# Day 3 — Image embeddings (CLIP) + save vectors

**Goal:** load clean dataset, compute CLIP image embeddings, and save `image_embeddings_*.npy`.

In [ ]:
!pip -q install pandas tqdm transformers torch pillow

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Load clean dataset

In [ ]:

df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
print("Rows:", df.shape[0])
print(df.columns)
print("Sample image path:", df.loc[0, "image_path"])


## 2) Load CLIP model + preprocess

In [ ]:

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

model.eval()
print("Device:", device, "Model:", model_name)


## 3) Compute embeddings (normalize to unit length)

In [ ]:

def embed_images(paths, batch_size=32):
    embs = []
    for i in tqdm(range(0, len(paths), batch_size)):
        batch_paths = paths[i:i+batch_size]
        imgs = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            imgs.append(img)
        inputs = processor(images=imgs, return_tensors="pt")
        inputs = {k:v.to(device) for k,v in inputs.items()}
        with torch.no_grad():
            feats = model.get_image_features(**inputs)  # (B,512)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        embs.append(feats.cpu().numpy().astype("float32"))
    return np.vstack(embs)

paths = df["image_path"].tolist()
image_emb = embed_images(paths, batch_size=32)
print("Embeddings shape:", image_emb.shape)
print("Example vector norm:", float(np.linalg.norm(image_emb[0])))


## 4) Save embeddings

In [ ]:

out_path = f"{BASE}/image_embeddings_2696.npy"
np.save(out_path, image_emb)
print("Saved embeddings to:", out_path)


✅ **Day 3 deliverable:** `image_embeddings_2696.npy`